In [ ]:
import pandas as pd
import os
import numpy as np
from datetime import datetime
import ast
from concurrent.futures import ThreadPoolExecutor, as_completed
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
import glob
import copy
import pickle
import pandas as pd
import os
import numpy as np
from datetime import datetime
import ast
from googletrans import Translator
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
import matplotlib.pyplot as plt
from huggingface_hub import login
import pickle

In [ ]:
Annotated_cleaned = **Data**

In [ ]:
year_to_prior = {}
current_prior = {}
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start Building Prior Knowledge")
for year in sorted(Annotated_cleaned["Year"].unique()):
    year_rows = Annotated_cleaned[Annotated_cleaned["Year"] == year]
    year_to_prior[year] = copy.deepcopy(current_prior)  # store snapshot before update
    for row_idx in range(year_rows.shape[0]):
        for fen,next_fen in year_rows.iloc[row_idx].Fen_Pairs:
            if fen not in current_prior.keys(): # If not exist, create such dictionary
                current_prior[fen]={next_fen:1}
            else:
                fen_dict = current_prior[fen]
                if next_fen not in fen_dict.keys():
                    current_prior[fen][next_fen]=1
                else:
                    current_prior[fen][next_fen]+=1
    if year % 25 == 0:
        print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Currently at {year}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Finish Building Prior Knowledge")

In [ ]:
len(current_prior.keys())

Store Embeddings into one file

In [ ]:
moves_embed = np.load(folder_path,allow_pickle=True)

In [ ]:
fens_embed = np.load(folder_path,allow_pickle=True)

In [ ]:
fens_lookup = np.load(folder_path,allow_pickle=True)

# Compute Surprise

In [ ]:
import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin
import argparse
import numpy as np
import chess
import os

In [ ]:
class ChessEncoder(nn.Module, PyTorchModelHubMixin):
    def __init__(self, d_model=256, nhead=8, num_layers=6, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers
        self.dim_feedforward = dim_feedforward
        self.dropout_rate = dropout

        self.patch_embed = nn.Linear(1, d_model)
        self.turn_embed = nn.Embedding(2, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, 64, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model)
        )
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, board_state, turn):
        batch_size = board_state.size(0)
        x = board_state.view(batch_size, 64, 1).float() # Ensure float input
        x = self.patch_embed(x)
        x = x + self.pos_embed
        turn_emb = self.turn_embed(turn).unsqueeze(1).expand(-1, 64, -1)
        x = x + turn_emb
        # The encoder output itself is the sequence embedding
        x = self.transformer(x)
        x = self.layer_norm(x)
        return x

    def encode_position(self, board_matrix, turn_value):
        """Generates a single embedding vector for a board position and turn.
           This requires an adaptation or assumption about how to get a single vector
           from the sequence output (e.g., average pooling, CLS token if used).
           Here, we'll average the sequence output for simplicity.
        """
        self.eval() # Ensure model is in eval mode
        with torch.no_grad():
            # Prepare tensors
            board_tensor = torch.tensor(board_matrix, dtype=torch.float32).unsqueeze(0) # Add batch dim
            turn_tensor = torch.tensor([turn_value], dtype=torch.long)

            # Move tensors to the same device as the model parameters
            device = next(self.parameters()).device
            board_tensor = board_tensor.to(device)
            turn_tensor = turn_tensor.to(device)

            # Get sequence embedding from forward pass
            sequence_embedding = self.forward(board_tensor, turn_tensor)

            # Pool the sequence embedding (e.g., mean pooling)
            # Exclude positional embedding if needed, or pool across the 64 squares
            pooled_embedding = torch.mean(sequence_embedding, dim=1) # Pool across the sequence length (64)

            return pooled_embedding.squeeze(0).cpu().numpy() # Remove batch dim and move to CPU

class ChessBoardHelper:
    """Helper class to manage chess board state and conversion."""
    def __init__(self):
        self.piece_values = {
            'P': 1, 'N': 2, 'B': 3, 'R': 4, 'Q': 5, 'K': 6,
            'p': -1, 'n': -2, 'b': -3, 'r': -4, 'q': -5, 'k': -6
        }
        self.board = chess.Board()

    def set_fen(self, fen: str):
        """Set board state from FEN string."""
        try:
            self.board = chess.Board(fen)
        except ValueError as e:
            print(f"Error: Invalid FEN string: {fen} - {e}")
            raise

    def get_matrix(self) -> np.ndarray:
        """Get the board state as an 8x8 numpy matrix suitable for the model."""
        matrix = np.zeros((8, 8), dtype=np.float32)
        for square in chess.SQUARES:
            piece = self.board.piece_at(square)
            if piece is not None:
                rank = chess.square_rank(square)
                file = chess.square_file(square)
                symbol = piece.symbol()
                matrix[rank, file] = self.piece_values[symbol]
        # The model expects a flattened (64,) or (batch, 64, 1) input typically
        # Flattening it here to match potential input expectations
        return matrix.flatten() # Return shape (64,)

    def get_turn_value(self) -> int:
        """Returns 0 for White's turn, 1 for Black's turn."""
        return 0 if self.board.turn == chess.WHITE else 1

In [ ]:
def main(model,fen_string):
    """Downloads model, processes FEN, generates and prints embedding."""
    # print(f"Loading ChessEncoder model from: {repo_id}")
    try:
        # Use PyTorchModelHubMixin's from_pretrained to download/load
        # model = ChessEncoder.from_pretrained(repo_id)
        model.eval() # Set to evaluation mode
        # print("Model loaded successfully.")
    except Exception as e:
        print(f"Error loading model from Hugging Face Hub: {e}")
        print("Please ensure the repository ID is correct and you have internet access.")
        print("Also ensure 'huggingface_hub' and 'torch' are installed.")
        return

    # Process FEN
    board_helper = ChessBoardHelper()
    try:
        board_helper.set_fen(fen_string)
        board_matrix = board_helper.get_matrix() # Shape (64,)
        turn_value = board_helper.get_turn_value()
        # print(f"Processing FEN: {fen_string}")
        # print(f"Turn: {'White' if turn_value == 0 else 'Black'}")
    except ValueError:
        # Error already printed in set_fen
        return
    except Exception as e:
        print(f"Error processing FEN string: {e}")
        return

    # Generate Embedding using the model's encode method
    try:
        # print("Generating embedding...")
        # Pass the flattened board matrix and turn value
        embedding = model.encode_position(board_matrix, turn_value)
        # print("\nGenerated Embedding:")
        # print(embedding)
        # print(f"\nEmbedding Dimension: {embedding.shape}")
        return embedding

    except Exception as e:
        print(f"Error during embedding generation: {e}")

In [ ]:
EMBED_DIM=256
MODEL_ID = "odestorm1/chesslm"
model = ChessEncoder.from_pretrained(MODEL_ID)

In [ ]:
def compute_surprise(i,row_dict, year_to_prior,BASELINE_YEAR,fens_lookup):
    year = row_dict["Year"]
    pairs = row_dict["Fen_Pairs"]
    playcount = row_dict["PlayCount"]
    prior_knowledge = year_to_prior.get(year)

    if year < BASELINE_YEAR:  # pre-year baseline
        return i,np.zeros(playcount,dtype=float)
    else:
        surprise = np.zeros(playcount,dtype=float)
        for move_idx in range(playcount):
            fen = pairs[move_idx][0] # Current State
            next_fen = pairs[move_idx][1] #Next State 
            # move_embed = fens_lookup[next_fen]-fens_lookup[fen] 
            if fen not in prior_knowledge.keys():
                try:
                    surprise[move_idx] = np.linalg.norm(fens_lookup[fen])
                except:
                    embedding = main(model, fen)
                    surprise[move_idx] = np.linalg.norm(embedding)
                    fens_lookup[fen] = embedding
                # print(f'{move_idx} fen does not exist in prior knowledge ')
            else:
                potential_fens = prior_knowledge[fen].keys()
                # print(potential_fens)
                # print(next_fen)
                if next_fen not in potential_fens:
                    distances = []
                    weights =[]
                    total_values = sum(prior_knowledge[fen].values())
                    for neighbor in potential_fens:
                        try:
                            label_embedding = fens_lookup[next_fen]
                        except:
                            label_embedding = main(model, next_fen)
                            fens_lookup[next_fen] = label_embedding
                        try:
                            possible_embedding = fens_lookup[neighbor]
                        except:
                            possible_embedding = main(model, neighbor)
                            fens_lookup[neighbor] = possible_embedding
                        distances.append(np.linalg.norm(label_embedding - possible_embedding))
                        weights.append(prior_knowledge[fen][neighbor]/total_values)
                    surprise[move_idx]= np.average(distances, 
                                            weights=weights)
                # else:
                #     print(f'{move_idx} should be 0 because it has existed')
            
    return i, surprise

In [ ]:
number_size=Annotated_cleaned.shape[0]
range_start = 0
BASELINE_YEAR=1960
range_end = min(range_start+number_size,Annotated_cleaned.shape[0])
N = min(number_size, Annotated_cleaned.shape[0]-range_start)

In [ ]:
surprise = np.zeros(N, dtype=object)
max_workers = 20  # tune to your CPU cores

print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Parallel computation starts")
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {
        ex.submit(compute_surprise, i,
                   {
                "Year":  Annotated_cleaned.iloc[i].Year,
                "Fen_Pairs":  Annotated_cleaned.iloc[i].Fen_Pairs,
                "PlayCount":  Annotated_cleaned.iloc[i].PlayCount,
                },
                year_to_prior,
                BASELINE_YEAR,
                fens_lookup): i
        for i in range(range_start,range_end)
    }
    completed = 0
    for fut in as_completed(futures):
        i,row_surprise = fut.result()
        surprise[i-range_start]  = row_surprise
        completed += 1
        if completed % 1000 == 0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{len(Annotated_cleaned)}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")